# Tesseract ? Kaggle CPU

Import this notebook directly into Kaggle. Select **Accelerator: None (CPU)** and **Internet: On**, then **Run All**. No project files need to be uploaded. The default `MODE="pilot"` runs the same 5 text + 5 nontext pages, seed 42. Inspect the OCR transcript, coverage and errors; then change `MODE="full"` in the first code cell and Run All for 9,000 pages. `FULL_SLICE` optionally limits the full run.

The notebook installs the Tesseract CLI only if missing and uses an isolated Python environment/kernel. It does not install torch, torchvision, Paddle, EasyOCR or pytesseract. All private Python processes (pip, import probes, kernel and OCR worker) use `-I -S` through an explicit venv launcher, skipping sitecustomize and .pth hooks entirely; dependencies use binary wheels, fixed direct versions and `pip check`. A real `OCR TEST 123` preflight runs before OPRB download. It retains English `tessdata_fast` at the local notebook's revision, OEM 1, PSM 3, automatically selected CPU workers/threads, one page at a time per worker, raw transcripts and the same GT/metrics/aggregation.

**Output:** `/kaggle/working/oprb_results/`. Every page commits to SQLite. Errors remain errors and are retried on the next run; empty successful transcripts are scored. The inference cell exports `tesseract_checkpoints.zip` in `finally`, including on interruption. Save a Kaggle version with outputs and download the ZIP before ending the session. A deleted host cannot execute cleanup or preserve unsaved working files.

**Resume:** attach your latest extracted Kaggle Tesseract SQLite as Input; set `RESUME_FROM` to the directory containing ALL worker SQLite files and `tesseract_cpu_tuning.json`. Config, CPU, Python/library versions, native Tesseract version and IDs must match. Working progress is never overwritten. Retain original filenames. Do not resume the Windows pilot or combine its timing with Kaggle. If the runtime changes, use a new configuration and pilot; no forced hash/name changes. Rerunning within the same session automatically skips successful pages.

Internet, apt mirrors, Hugging Face availability and Kaggle session limits can still cause failures. Diagnostics remain visible; no notebook can guarantee zero errors on an external runtime. Validate this revision with a real Kaggle pilot before a long run.

References: [Tesseract installation](https://tesseract-ocr.github.io/tessdoc/Installation.html), [pip interpreter isolation](https://pip.pypa.io/en/stable/topics/python-option/), [OPRB](https://huggingface.co/datasets/kpurkayastha/OPRB).

Startup repair: private IPython/Jupyter configuration directories prevent inherited Kaggle profiles/extensions. Matplotlib and its inline backend are pinned and tested with the noninteractive Agg backend; no warning filters are used. Look for `[STARTUP OK] isolated kernel, wrapt, matplotlib`. Existing checkpoint configuration may change because matplotlib packages/pip are now recorded; retain old SQLite names and do not force resume. Save current checkpoints before importing this revision and restarting the session.

CPU throughput tuning: detect CPU affinity and cgroup quota. On a four-logical-CPU host compare 1x1, 1x2, 1x4, 2x1, 2x2 and 4x1 (workers x threads), restricting workers*threads to the detected budget. Benchmark the same 12 tasks (3 seed-42 pilot images repeated four times, 2 text:1 nontext) after all worker warm-ups. Select by wall pages/s only, never GT/accuracy. This small sample does not prove a globally optimal profile. Freeze tesseract_cpu_tuning.json; pilot -> full reuses it when CPU/runtime/reader/calibration match. Save it with all worker SQLite files. Old one-worker checkpoints remain separate configurations.

Each worker owns one SQLite. Global page-index modulo worker count keeps pilot/full/slices disjoint. A supervisor relays logs, reports total throughput and ETA every 15 seconds, and retries crashes/stalls at most twice from committed progress. It stops after an 11-hour inference budget and exports partial progress; Kaggle host/session limits still apply. Native children are terminated with their process group on interrupt. Multiworker pages.seconds measures processing under shared CPU contention, not independent single-worker latency. Mean seconds divided by workers is only an approximate wall-time projection.

Threading reference: [Tesseract CPU performance FAQ](https://tesseract-ocr.github.io/tessdoc/FAQ.html#can-i-increase-speed-of-ocr).


In [ ]:
from pathlib import Path
import sys, os, subprocess, tempfile, json
MODE = "pilot"
FULL_SLICE = slice(None)
if not Path("/kaggle/working").is_dir():
    raise RuntimeError("Open this notebook on Kaggle.")
OUTPUT = Path("/kaggle/working/oprb_results")
GPU_WORKERS = 1  # CPU calibration selects OCR parallelism.
RESUME_FROM = None  # Optional: directory with latest tuned worker SQLite files + tuning JSON.
# Keep Kaggle's startup hooks and user packages out of every private process.
runtime_env = {k: v for k, v in os.environ.items()
               if k not in {"PYTHONPATH", "PYTHONHOME", "PYTHONSTARTUP", "PYTHONUSERBASE", "PYTHONINSPECT", "LD_LIBRARY_PATH", "LD_PRELOAD", "TESSDATA_PREFIX"}}
runtime_env.update(PYTHONNOUSERSITE="1", PYTHONUNBUFFERED="1", USE_TF="0", HF_HUB_DISABLE_XET="1", HF_HUB_DISABLE_TELEMETRY="1",
                   MKL_NUM_THREADS="2", OPENBLAS_NUM_THREADS="2", NUMEXPR_NUM_THREADS="2",
                   CUDA_VISIBLE_DEVICES="", OMP_NUM_THREADS="2", OMP_THREAD_LIMIT="2",
                   OMP_MAX_ACTIVE_LEVELS="1", OMP_NESTED="FALSE", OMP_DYNAMIC="FALSE", MKL_DYNAMIC="FALSE")


def run_logged(args, **kwargs):
    """Forward subprocess output to this notebook and stop children on interruption."""
    proc = subprocess.Popen(args, start_new_session=(os.name == "posix"), stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, encoding="utf-8", errors="replace", bufsize=1, **kwargs)
    try:
        for line in proc.stdout:
            print(line, end="", flush=True)
        returncode = proc.wait()
        if returncode:
            raise subprocess.CalledProcessError(returncode, args)
    finally:
        if proc.poll() is None:
            if os.name == "posix":
                import signal
                os.killpg(proc.pid, signal.SIGTERM)
            else:
                proc.terminate()
            try:
                proc.wait(timeout=10)
            except subprocess.TimeoutExpired:
                if os.name == "posix":
                    os.killpg(proc.pid, signal.SIGKILL)
                else:
                    proc.kill()
                proc.wait()
        proc.stdout.close()


import venv
from jupyter_client import KernelManager
from IPython.core.magic import register_cell_magic
# Install only the system CLI if missing. Never upgrade Kaggle's Python packages.
import shutil, importlib.util
exe = "/usr/bin/tesseract" if Path("/usr/bin/tesseract").is_file() else shutil.which("tesseract")
if not exe:
    prefix = [] if os.geteuid() == 0 else ["sudo", "-n"]
    apt_env = dict(runtime_env, DEBIAN_FRONTEND="noninteractive")
    run_logged(prefix + ["apt-get", "-o", "Acquire::Retries=3", "update"], env=apt_env)
    run_logged(prefix + ["apt-get", "-o", "Acquire::Retries=3", "install", "-y", "--no-install-recommends", "tesseract-ocr"], env=apt_env)
    exe = shutil.which("tesseract")
if not exe:
    raise RuntimeError("Tesseract CLI unavailable; inspect apt output above.")
run_logged([exe, "--version"], env=runtime_env)
runtime = Path(tempfile.gettempdir()) / ("oprb-private-tesseract-v2-" + sys.implementation.cache_tag)
python = runtime / "bin/python"
if not python.exists():
    # Run venv itself in isolation so injected sitecustomize/PYTHONPATH cannot leak in.
    run_logged([sys.executable, "-I", "-S", "-m", "venv", "--without-pip", str(runtime)], env=runtime_env)
# Bootstrap pip by copying its self-contained vendored package into the venv.
# pip --python can spawn an interpreter before wrapt has been installed.
pip_spec = importlib.util.find_spec("pip")
if pip_spec is None:
    raise RuntimeError("Kaggle's base pip is unavailable. Restart a standard Kaggle Python session.")
private_site = runtime / f"lib/python{sys.version_info.major}.{sys.version_info.minor}/site-packages"
private_site.mkdir(parents=True, exist_ok=True)
shutil.copytree(Path(pip_spec.origin).parent, private_site / "pip", dirs_exist_ok=True)
from importlib.metadata import distribution
pip_dist = distribution("pip")
for item in pip_dist.files or []:
    if item.parts[0].endswith(".dist-info"):
        metadata_dir = Path(pip_dist.locate_file(item.parts[0]))
        shutil.copytree(metadata_dir, private_site / metadata_dir.name, dirs_exist_ok=True)
        break
private_launcher_code = 'import os, runpy, sys\nfrom pathlib import Path\nprefix = Path(sys.argv.pop(1)).resolve()\n# Python 3.12/3.13 normally set the venv prefix in site.py, which -S skips.\nsys.prefix = sys.exec_prefix = str(prefix)\nsite_path = prefix / ("Lib/site-packages" if os.name == "nt" else f"lib/python{sys.version_info.major}.{sys.version_info.minor}/site-packages")\nsys.path.insert(0, str(site_path))\n# Do not call site.main/addsitedir: neither .pth nor sitecustomize should execute.\nmode = sys.argv[1]\nif mode == "-m":\n    module = sys.argv[2]\n    sys.argv = sys.argv[2:]\n    runpy.run_module(module, run_name="__main__", alter_sys=True)\nelif mode == "-c":\n    code = sys.argv[2]\n    sys.argv = ["-c", *sys.argv[3:]]\n    exec(compile(code, "<private-runtime>", "exec"), {"__name__": "__main__"})\nelse:\n    raise ValueError(f"Unsupported private launcher mode: {mode}")\n'
launcher = runtime / "launch_private.py"
launcher.write_text(private_launcher_code, encoding="utf-8")
private_python = [str(python), "-I", "-S", str(launcher), str(runtime)]
# Isolate IPython profiles/startup scripts and Jupyter config as well as Python paths.
for key, name in [("IPYTHONDIR", "ipython"), ("JUPYTER_CONFIG_DIR", "jupyter"), ("MPLCONFIGDIR", "matplotlib")]:
    folder = runtime / "config" / name
    folder.mkdir(parents=True, exist_ok=True)
    runtime_env[key] = str(folder)
runtime_env["MPLBACKEND"] = "Agg"
pip = private_python + ["-m", "pip", "--isolated"]
requirements = ["jiwer==4.0.0", "numpy==2.2.6", "pandas==2.2.3", "pillow==11.3.0",
                "huggingface-hub==0.36.0", "ipykernel==6.30.1", "wrapt==1.17.3",
                "matplotlib==3.10.7", "matplotlib-inline==0.1.7"]
run_logged(pip + ["install", "--only-binary=:all:", "--retries", "3", "--timeout", "60", *requirements], env=runtime_env)
run_logged(pip + ["check"], env=runtime_env)
probe_code = """import sys,sqlite3,wrapt,numpy,pandas,PIL,jiwer,huggingface_hub,ipykernel
import matplotlib, matplotlib_inline
assert sys.flags.no_site and "sitecustomize" not in sys.modules
matplotlib.use("Agg")
from matplotlib import pyplot as plt
fig = plt.figure(); plt.close(fig)
from importlib.metadata import version
assert jiwer.cer('abc','abc') == 0
assert pandas.Series(numpy.array([1.,3.])).mean() == 2
print('Runtime OK:', sys.version.split()[0], {p:version(p) for p in ['numpy','pandas','pillow','jiwer','huggingface-hub','ipykernel','wrapt','matplotlib','matplotlib-inline']}, flush=True)
"""
run_logged(private_python + ["-c", probe_code], env=runtime_env)
if "manager" in globals() and manager.has_kernel:
    client.stop_channels()
    manager.shutdown_kernel(now=True)
manager = KernelManager(kernel_name="python3")
manager.kernel_spec.argv = private_python + ["-m", "ipykernel_launcher", "-f", "{connection_file}", "--IPKernelApp.matplotlib=agg"]
manager.start_kernel(env=runtime_env)
client = manager.blocking_client()
client.start_channels()
client.wait_for_ready(timeout=120)

def show_output(message):
    from IPython.display import display
    content = message["content"]
    if message["msg_type"] == "stream":
        print(content["text"], end="", flush=True)
    elif message["msg_type"] in {"display_data", "execute_result"}:
        display(content["data"], raw=True)
    elif message["msg_type"] == "error":
        print("\n".join(content["traceback"]), flush=True)

@register_cell_magic
def benchmark(line, cell):
    try:
        reply = client.execute_interactive(cell, timeout=None, output_hook=show_output)
    except KeyboardInterrupt:
        manager.interrupt_kernel()
        raise
    if reply["content"]["status"] != "ok":
        raise RuntimeError(reply["content"].get("evalue", "Private kernel failed"))

benchmark("", f"from pathlib import Path\nimport os, sys, subprocess, tempfile, json\n"
          f"assert sys.prefix == {str(runtime)!r}\nMODE={MODE!r}\nFULL_SLICE={FULL_SLICE!r}\n"
          f"OUTPUT=Path({str(OUTPUT)!r})\nGPU_WORKERS={GPU_WORKERS}")
benchmark("", "runtime_env = dict(os.environ)")
import inspect
benchmark("", inspect.getsource(run_logged))

benchmark("", f"EXE={exe!r}\nRESUME_FROM={str(RESUME_FROM) if RESUME_FROM else None!r}")

benchmark("", f"private_python={private_python!r}\nassert sys.flags.no_site and 'sitecustomize' not in sys.modules\nimport matplotlib, matplotlib_inline, wrapt\nprint('[STARTUP OK] isolated kernel, wrapt, matplotlib', flush=True)")


In [ ]:
%%benchmark
import hashlib, shutil, time, urllib.error, urllib.request
CACHE = Path(tempfile.gettempdir()) / "oprb_test"
CACHE.mkdir(parents=True, exist_ok=True)

def retry_download(function):
    for attempt in range(3):
        try:
            return function()
        except Exception as exc:
            response = getattr(exc, "response", None)
            status = getattr(response, "status_code", getattr(exc, "code", None))
            transient = status in {408, 429, 500, 502, 503, 504}
            if status is None:
                import requests
                transient = isinstance(exc, (OSError, TimeoutError, requests.exceptions.ConnectionError,
                                             requests.exceptions.Timeout))
            if not transient or attempt == 2:
                raise
            print(f"Temporary download failure; retry {attempt+1}/2: {exc}", flush=True)
            time.sleep(2 ** attempt)

from pathlib import Path
weights = {}
revisions = {}
import urllib.request
exe = EXE
rev = "87416418657359cb625c412a48b6e1d6d41c29bd"
revisions["tessdata_fast"] = rev
folder = CACHE / rev
folder.mkdir(parents=True, exist_ok=True)
path = folder / "eng.traineddata"
expected_sha = '7d4322bd2a7749724879683fc3912cb542f19906c83bcc1a52132556427170b2'
if not path.exists() or hashlib.sha256(path.read_bytes()).hexdigest() != expected_sha:
    url = f"https://raw.githubusercontent.com/tesseract-ocr/tessdata_fast/{rev}/eng.traineddata"
    data = retry_download(lambda: urllib.request.urlopen(url, timeout=90).read())
    if hashlib.sha256(data).hexdigest() != expected_sha:
        raise ValueError("Downloaded tessdata_fast checksum mismatch")
    tmp = path.with_suffix(".part")
    tmp.write_bytes(data)
    tmp.replace(path)
weights["eng"] = str(folder)

def load_reader(model, weights, prompt):
    def read(path):
        result = subprocess.run([weights["exe"], str(path), "stdout", "--tessdata-dir", weights["eng"],
                                 "-l", "eng", "--oem", "1", "--psm", "3"],
                                check=True, capture_output=True, encoding="utf8", timeout=180)
        return result.stdout, False
    return read


weight_files = [p for folder in weights.values() for p in Path(folder).rglob("*")
                if p.suffix in {".safetensors", ".pdiparams", ".pt", ".pth", ".traineddata"}]
assert weight_files, "Inspect weight filenames for this model"
weight_mib = sum(p.stat().st_size for p in weight_files) / 2**20
weights["exe"] = exe
# Real OCR preflight, before downloading the large OPRB archives.
from PIL import Image, ImageDraw, ImageFont
with tempfile.TemporaryDirectory() as td:
    image = Image.new("RGB", (900, 180), "white")
    font_path = Path("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf")
    font = ImageFont.truetype(str(font_path), 64) if font_path.is_file() else ImageFont.load_default(size=64)
    ImageDraw.Draw(image).text((35, 45), "OCR TEST 123", fill="black", font=font)
    smoke_path = Path(td) / "smoke.png"
    image.save(smoke_path)
    text, _ = load_reader("tesseract", weights, "")(smoke_path)
    print("Tesseract OCR preflight:", repr(text), flush=True)
    if "OCR TEST 123" not in " ".join(text.split()).upper():
        raise RuntimeError("OCR preflight failed; inspect transcript and native library diagnostics above.")


In [ ]:
%%benchmark
MODEL = 'tesseract'
IS_GPU = False
model_python = sys.executable
OUTPUT.mkdir(parents=True, exist_ok=True)
assert MODE in {"pilot","full"}
CACHE = Path(tempfile.gettempdir()) / "oprb_test"
import hashlib, json, random, shutil, sqlite3, tarfile, time, unicodedata
from collections import Counter
from importlib.metadata import version
from huggingface_hub import HfApi, hf_hub_download, snapshot_download
from PIL import Image
from jiwer import cer, wer
import pandas as pd

REPO = "kpurkayastha/OPRB"
DATA_REV = 'c7ab9fdf2bf33f297bba62e4842d3e5094d11ec2'
ROOT = CACHE / DATA_REV
COUNTS = {"text": 6000, "nontext": 3000}
ARCHIVES = {'text/img': '64c0f0fe5df6caf988379602e320563367ad2865bd964109f6985bf61d8c396c', 'text/gt': '09a95490834198959ad253db0ed3c9ffab4aa289aef059a6252173a67eaca901', 'nontext/img': '95001b92c371ad6a1a2440bde1cbb5cf19e1ad5587f7ab6725fed085a1fa96a4', 'nontext/gt': 'b15fe672eb26000ad9cd52a1bcaf5a468405392ee0789334eb3dc72c2bf2f000'}
pages = []
for region, count in COUNTS.items():
    for kind, suffix in [("img", ".jpg"), ("gt", ".txt")]:
        target = ROOT / region / kind
        target.mkdir(parents=True, exist_ok=True)
        if not (target / ".complete").is_file():
            archive = retry_download(lambda: hf_hub_download(REPO, f"test/{region}/{kind}.tar.gz", repo_type="dataset", revision=DATA_REV))
            expected = ARCHIVES[f"{region}/{kind}"]
            with open(archive, "rb") as stream:
                if hashlib.file_digest(stream, "sha256").hexdigest() != expected:
                    raise ValueError(f"Archive checksum mismatch: {region}/{kind}")
            seen = set()
            with tarfile.open(archive, "r:gz") as tar:
                for member in tar:
                    name = Path(member.name).name
                    if not member.isfile() or Path(name).suffix.lower() != suffix:
                        continue
                    if name in seen:
                        raise ValueError(f"Repeated basename: {name}")
                    seen.add(name)
                    with tar.extractfile(member) as src, (target / name).open("wb") as dest:
                        shutil.copyfileobj(src, dest)
            assert len(seen) == count, (region, kind, len(seen))
            (target / ".complete").touch()
    imgs = {p.stem for p in (ROOT / region / "img").glob("*.jpg")}
    refs = {p.stem for p in (ROOT / region / "gt").glob("*.txt")}
    assert imgs == refs and len(imgs) == count, (region, len(imgs), len(refs))
    pages.extend((region, stem) for stem in sorted(imgs))
selected = pages[FULL_SLICE] if MODE == "full" else [
    key for region in COUNTS
    for key in random.Random(42).sample([p for p in pages if p[0] == region], 5)
]
print(DATA_REV, Counter(r for r, _ in selected))

In [ ]:
%%benchmark
def reference(region, stem):
    tokens = []
    path = ROOT / region / "gt" / (stem + ".txt")
    for line in path.read_text(encoding="utf-8-sig").splitlines():
        if not line.strip():
            continue
        fields = line.split("\t")
        if len(fields) not in (10, 11):
            raise ValueError(f"Malformed GT row in {path.name}: {line[:60]!r}")
        if fields[0].strip():
            tokens.append(fields[0].strip())
    if not tokens:
        raise ValueError(f"Empty GT: {path.name}")
    return " ".join(tokens)

def normalize(s):
    return " ".join(unicodedata.normalize("NFKC", s).casefold().split())

def metrics(ref, hyp):
    ref, hyp = normalize(ref), normalize(hyp)
    a, b = Counter(ref.split()), Counter(hyp.split())
    hits = sum((a & b).values())
    p = hits / sum(b.values()) if b else 0
    r = hits / sum(a.values())
    return cer(ref, hyp), wer(ref, hyp), p, r, 2*p*r/(p+r) if hits else 0

assert metrics("Ａ b", "a B") == (0, 0, 1, 1, 1)
assert metrics("a", "") == (1, 1, 0, 0, 0)
assert metrics("a a", "a")[2:] == (1, .5, 2/3)
assert metrics("a b", "b a")[1:] == (1, 1, 1, 1)
# Validate all annotations and image files before the inference loop; no GT enters OCR.
for r, s in pages:
    reference(r, s)
    with Image.open(ROOT / r / "img" / (s + ".jpg")) as image:
        image.verify()
print("Validated 6,000 text + 3,000 nontext image/GT pairs", flush=True)


In [ ]:
%%benchmark
from contextlib import closing

def restore_checkpoint(source, output, config, pages, worker=0):
    if not source:
        return
    source, output = Path(source).resolve(), Path(output)
    digest = hashlib.sha256(json.dumps(config, sort_keys=True).encode()).hexdigest()[:16]
    filename = f"tesseract-{digest}-{worker}.sqlite"
    if source.name != filename:
        raise ValueError("Resume filename/config differs from this Kaggle run; retain old results separately.")
    allowed = set(pages)
    with closing(sqlite3.connect(source.as_uri() + "?mode=ro", uri=True)) as incoming:
        if incoming.execute("PRAGMA integrity_check").fetchall() != [("ok",)]:
            raise ValueError("Invalid resume SQLite")
        configs = incoming.execute("SELECT json FROM config").fetchall()
        if len(configs) != 1 or json.loads(configs[0][0]) != config:
            raise ValueError("Resume configuration mismatch")
        incoming_rows = incoming.execute("SELECT region,stem,prediction,seconds,truncated,error FROM pages").fetchall()
        if any(row[:2] not in allowed for row in incoming_rows):
            raise ValueError("Resume contains unexpected page IDs")
        target = output / filename
        if target.exists():
            with closing(sqlite3.connect(target.resolve().as_uri() + "?mode=ro", uri=True)) as current:
                if current.execute("PRAGMA integrity_check").fetchall() != [("ok",)]:
                    raise ValueError("Invalid working SQLite")
                configs = current.execute("SELECT json FROM config").fetchall()
                if len(configs) != 1 or json.loads(configs[0][0]) != config:
                    raise ValueError("Working configuration mismatch")
                rows = {row[:2]: row for row in current.execute("SELECT region,stem,prediction,seconds,truncated,error FROM pages")}
                if any(row[-1] is None and rows.get(row[:2]) != row for row in incoming_rows):
                    raise ValueError("Input has successes absent/different in working DB. Use a new OUTPUT to restore this input safely.")
            print("[RESUME] Keeping working checkpoint", target, flush=True)
        else:
            stage = target.with_suffix(".restoring")
            try:
                with closing(sqlite3.connect(stage)) as restored:
                    incoming.backup(restored)
                stage.replace(target)
            finally:
                stage.unlink(missing_ok=True)
        print("[RESUME] CONFIG MATCHED", filename, flush=True)

def export_checkpoints(output):
    import zipfile
    output = Path(output)
    archive = output / "tesseract_checkpoints.zip"
    staged = archive.with_suffix(".tmp")
    with tempfile.TemporaryDirectory() as td, zipfile.ZipFile(staged, "w", zipfile.ZIP_DEFLATED) as bundle:
        for path in sorted(output.glob("tesseract-*.sqlite")):
            snapshot = Path(td) / path.name
            with closing(sqlite3.connect(path.resolve().as_uri() + "?mode=ro", uri=True)) as original:
                with closing(sqlite3.connect(snapshot)) as copied:
                    original.backup(copied)
            bundle.write(snapshot, path.name)
        for name in ["tesseract_cpu_tuning.json", "session_status.json"]:
            if (output/name).is_file():
                bundle.write(output/name, name)
    staged.replace(archive)
    print("Checkpoint ZIP:", archive, flush=True)

def evaluate(job):
    import hashlib, json, os, platform, sqlite3, time
    from pathlib import Path
    model, region_pages = job["model"], job["pages"]
    sync = lambda: None
    if job["gpu"]:
        import torch
        if not torch.cuda.is_available():
            raise RuntimeError("Select a CUDA GPU runtime or reduce GPU_WORKERS.")
        device = torch.cuda.get_device_name()
        sync = torch.cuda.synchronize
    else:
        device = "CPU"
    cfg = dict(job["config"], device=device)
    key = hashlib.sha256(json.dumps(cfg, sort_keys=True).encode()).hexdigest()
    db = sqlite3.connect(Path(job["output"]) / f"{model}-{key[:16]}-{job['worker']}.sqlite")
    db.execute("CREATE TABLE IF NOT EXISTS config (json TEXT PRIMARY KEY)")
    db.execute("INSERT OR IGNORE INTO config VALUES (?)", (json.dumps(cfg, sort_keys=True),))
    db.execute("""CREATE TABLE IF NOT EXISTS pages (
        region TEXT, stem TEXT, prediction TEXT, seconds REAL, truncated INTEGER, error TEXT,
        PRIMARY KEY(region, stem))""")
    db.commit()
    done = set(db.execute("SELECT region, stem FROM pages WHERE error IS NULL"))
    pending = [(r,s) for r,s in region_pages if (r,s) not in done]
    if pending:
        read = load_reader(model, job["weights"], job["prompt"])
        def path(r, s):
            return Path(job["root"]) / r / "img" / (s + ".jpg")
        read(path(*pending[0]))
        for i, (r, s) in enumerate(pending):
            try:
                sync()
                start = time.perf_counter()
                prediction, truncated = read(path(r, s))
                sync()
                row = (r, s, prediction, time.perf_counter()-start, int(truncated), None)
            except Exception as exc:
                row = (r, s, None, None, None, repr(exc) + " " + str(getattr(exc, "stderr", "")))
                print(r, s, repr(exc), flush=True)
            with db:
                db.execute("INSERT OR REPLACE INTO pages VALUES (?,?,?,?,?,?)", row)
            if (i+1) % 25 == 0 or i+1 == len(pending):
                print(model, job["worker"], i+1, "/", len(pending), flush=True)
    db.close()


def detect_cpu_budget():
    import math, os
    from pathlib import Path
    affinity_ids = sorted(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else list(range(os.cpu_count() or 1))
    quota = None
    try:
        fields = Path("/sys/fs/cgroup/cpu.max").read_text().split()
        if fields[0] != "max":
            quota = int(fields[0]) / int(fields[1])
    except (OSError, ValueError, IndexError, ZeroDivisionError):
        try:
            limit = int(Path("/sys/fs/cgroup/cpu/cpu.cfs_quota_us").read_text())
            period = int(Path("/sys/fs/cgroup/cpu/cpu.cfs_period_us").read_text())
            if limit > 0:
                quota = limit / period
        except (OSError, ValueError, ZeroDivisionError):
            pass
    budget = max(1, min(len(affinity_ids), math.floor(quota) if quota else len(affinity_ids)))
    cores = set()
    for cpu in affinity_ids:
        try:
            root = Path(f"/sys/devices/system/cpu/cpu{cpu}/topology")
            cores.add((int((root/"physical_package_id").read_text()), int((root/"core_id").read_text())))
        except (OSError, ValueError):
            pass
    return dict(affinity_cpus=affinity_ids, quota_cpus=quota, physical_cores=len(cores) or None, budget=budget)

def cpu_candidates(budget):
    return [dict(workers=w, threads=t) for w,t in [(1,1),(1,2),(1,4),(2,1),(2,2),(4,1)] if w*t <= budget]

def stop_cpu_process(proc):
    import os, signal, subprocess
    # Kill descendants even if a crashed Python group leader has already exited.
    if os.name == "posix" or proc.poll() is None:
        try:
            os.killpg(proc.pid, signal.SIGTERM) if os.name == "posix" else proc.terminate()
        except ProcessLookupError:
            pass
        try:
            proc.wait(timeout=10)
        except subprocess.TimeoutExpired:
            try:
                os.killpg(proc.pid, signal.SIGKILL) if os.name == "posix" else proc.kill()
            except ProcessLookupError:
                pass
            proc.wait()

def cpu_process_env(base, threads):
    env = dict(base)
    env.update(CUDA_VISIBLE_DEVICES="", USE_TF="0", USE_FLAX="0", TOKENIZERS_PARALLELISM="false",
               OMP_MAX_ACTIVE_LEVELS="1", OMP_NESTED="FALSE", OMP_DYNAMIC="FALSE", MKL_DYNAMIC="FALSE")
    for name in ("OMP_NUM_THREADS", "OMP_THREAD_LIMIT", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
        env[name] = str(threads)
    return env

def calibrate_worker(job):
    import json, time
    from pathlib import Path
    read = load_reader("tesseract", job["weights"], "")
    paths = [Path(job["root"]) / r / "img" / (s + ".jpg") for r, s in job["pages"]]
    read(paths[0])  # Model loading and warm-up are outside the shared wall timer.
    Path(job["ready"]).touch()
    while not Path(job["start"]).exists():
        time.sleep(0.02)
    elapsed = 0.0
    for path in paths:
        start = time.perf_counter()
        prediction, truncated = read(path)
        elapsed += time.perf_counter() - start
        if not isinstance(prediction, str):
            raise TypeError("The OCR reader must return text")
    Path(job["result"]).write_text(json.dumps(dict(pages=len(paths), seconds=elapsed)))

def launch_cpu_jobs(jobs, code, profile, base_env, timeout, barrier=None):
    import os, json, subprocess, tempfile, time
    from concurrent.futures import ThreadPoolExecutor
    from pathlib import Path
    processes, streams, futures = [], [], []
    pool = ThreadPoolExecutor(max_workers=len(jobs))
    start = None
    def relay(process, index):
        for line in process.stdout:
            print(f"[cpu {index}] {line}", end="", flush=True)
    try:
        for index, job in enumerate(jobs):
            stream = tempfile.TemporaryFile(mode="w+t", encoding="utf-8")
            streams.append(stream)
            stream.write(json.dumps(job))
            stream.seek(0)
            process = subprocess.Popen(private_python + ["-c", code], stdin=stream,
                                       start_new_session=(os.name == "posix"),
                                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                                       encoding="utf-8", errors="replace", bufsize=1,
                                       env=cpu_process_env(base_env, profile["threads"]))
            processes.append(process)
            futures.append(pool.submit(relay, process, index))
        if barrier is not None:
            deadline = time.monotonic() + timeout
            while not all(Path(job["ready"]).is_file() for job in jobs):
                for process in processes:
                    if process.poll() is not None:
                        raise RuntimeError(f"Calibration worker exited before ready: {process.returncode}")
                if time.monotonic() >= deadline:
                    raise TimeoutError("Calibration model initialization timed out")
                time.sleep(0.05)
            start = time.monotonic()
            Path(barrier).touch()
        else:
            start = time.monotonic()
        # Calibration is bounded. Full inference uses no artificial deadline.
        deadline = start + timeout if timeout is not None else None
        while True:
            statuses = [process.poll() for process in processes]
            failed = [code_return for code_return in statuses if code_return is not None and code_return != 0]
            if failed:
                raise RuntimeError(f"CPU worker failed with exit code {failed[0]}")
            if all(code_return is not None for code_return in statuses):
                break
            if deadline is not None and time.monotonic() >= deadline:
                raise TimeoutError("CPU calibration timed out during inference")
            time.sleep(0.02)
        elapsed = time.monotonic() - start
        for future in futures:
            future.result()
        return elapsed
    finally:
        for process in processes:
            stop_cpu_process(process)
        pool.shutdown(wait=True)
        for process in processes:
            process.stdout.close()
        for stream in streams:
            stream.close()

def select_cpu_profile(records):
    import math
    valid = [r for r in records if not r.get("error") and isinstance(r.get("pages_per_second"), (int, float))
             and math.isfinite(r["pages_per_second"]) and r["pages_per_second"] > 0]
    if not valid:
        raise RuntimeError("No CPU profile completed calibration successfully; inspect worker logs")
    return max(valid, key=lambda r: r["pages_per_second"])["profile"]

def checkpoint_progress(job):
    import hashlib, json, sqlite3
    from pathlib import Path
    from contextlib import closing
    cfg = dict(job["config"], device="CPU")
    key = hashlib.sha256(json.dumps(cfg, sort_keys=True).encode()).hexdigest()[:16]
    path = Path(job["output"]) / f"tesseract-{key}-{job['worker']}.sqlite"
    if not path.exists():
        return path, 0, 0
    try:
        with closing(sqlite3.connect(path.resolve().as_uri()+"?mode=ro", uri=True, timeout=5)) as db:
            rows = dict(((r,s),error) for r,s,error in db.execute("SELECT region,stem,error FROM pages"))
    except sqlite3.OperationalError as exc:
        if "no such table" in str(exc) or "locked" in str(exc):
            return path, 0, 0  # A worker may be creating the schema/committing its first page.
        raise
    keys = {tuple(p) for p in job["pages"]}
    return path, sum(k in rows and rows[k] is None for k in keys), sum(k in rows and rows[k] is not None for k in keys)

def supervise_cpu_jobs(jobs, code, profile, base_env, output, deadline, max_restarts=2, stall_seconds=900):
    import os, json, subprocess, tempfile, threading, time
    from pathlib import Path
    output = Path(output)
    output.mkdir(parents=True, exist_ok=True)
    states = []
    start = time.monotonic()
    def relay(proc, worker, logfile):
        with open(logfile, "a", encoding="utf-8") as log:
            for line in proc.stdout:
                log.write(line)
                log.flush()
                print(f"[cpu {worker}] {line}", end="", flush=True)
    def launch(state):
        stream = tempfile.TemporaryFile(mode="w+t", encoding="utf-8")
        stream.write(json.dumps(state["job"]))
        stream.seek(0)
        try:
            proc = subprocess.Popen(private_python + ["-c", code], stdin=stream,
                                     start_new_session=(os.name == "posix"),
                                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                                     encoding="utf-8", errors="replace", bufsize=1,
                                     env=cpu_process_env(base_env, profile["threads"]))
        finally:
            stream.close()
        state.update(proc=proc, last_change=time.monotonic(), marker=None)
        thread = threading.Thread(target=relay, args=(proc,state["job"]["worker"],
                                  str(output/f"worker-{state['job']['worker']}.log")), daemon=True)
        state["thread"] = thread
        thread.start()
    def finish(state, terminate=False):
        proc = state.get("proc")
        if proc is None:
            return
        stop_cpu_process(proc) if terminate else proc.wait()
        state["thread"].join(timeout=5)
        proc.stdout.close()
        state["proc"] = None
    initial_ok = sum(checkpoint_progress(job)[1] for job in jobs)
    result = "paused"
    last_print = 0
    try:
        for job in jobs:
            path, ok, errors = checkpoint_progress(job)
            state = dict(job=job, proc=None, restarts=0, done=ok==len(job["pages"]), ok=ok, errors=errors)
            states.append(state)
            if not state["done"] and time.monotonic() < deadline:
                launch(state)
        while True:
            now = time.monotonic()
            if now >= deadline:
                result = "paused_before_session_limit"
                break
            failed = False
            for state in states:
                if state["done"]:
                    continue
                path, ok, errors = checkpoint_progress(state["job"])
                state.update(ok=ok, errors=errors)
                marker = (path.stat().st_mtime_ns, path.stat().st_size) if path.exists() else None
                if marker != state["marker"]:
                    state.update(marker=marker,last_change=now)
                proc = state["proc"]
                exited = proc.poll() is not None
                stalled = now-state["last_change"] > stall_seconds
                if exited or stalled:
                    exitcode = proc.returncode
                    finish(state, terminate=stalled)
                    _, ok, errors = checkpoint_progress(state["job"])
                    state.update(ok=ok, errors=errors)
                    if ok == len(state["job"]["pages"]):
                        state["done"] = True
                    elif state["restarts"] < max_restarts:
                        state["restarts"] += 1
                        print(f"[RECOVER] worker={state['job']['worker']} exit={exitcode} stalled={stalled}; "
                              f"restart={state['restarts']}/{max_restarts}; successful pages preserved",flush=True)
                        launch(state)
                    else:
                        failed = True
                        state.update(done=True,failed=True)
                        print(f"[RECOVER] worker={state['job']['worker']} exhausted retries; healthy workers continue",flush=True)
            if now-last_print >= 15 or failed or all(s["done"] for s in states):
                ok = sum(s["ok"] for s in states)
                total = sum(len(s["job"]["pages"]) for s in states)
                rate = max(0, ok-initial_ok) / max(.001, now-start)
                eta = (total-ok)/rate/60 if rate else float("inf")
                print(f"[TOTAL] {ok}/{total} successful; errors={sum(s['errors'] for s in states)}; "
                      f"wall={(now-start)/60:.1f} min; {rate:.3f} pages/s; ETA={eta:.1f} min",flush=True)
                last_print = now
            if all(s["done"] for s in states):
                result = "partial_worker_failure" if any(s.get("failed") for s in states) else "complete_selected_pages"
                break
            time.sleep(1)
    except KeyboardInterrupt:
        result = "interrupted"
        raise
    finally:
        for state in states:
            finish(state, terminate=True)
        summary = dict(status=result, workers=[])
        for state in states:
            path, ok, errors = checkpoint_progress(state["job"])
            summary["workers"].append(dict(file=path.name,successful=ok,errors=errors,
                                           selected=len(state["job"]["pages"]),restarts=state["restarts"],failed=state.get("failed",False)))
        staged = output/"session_status.tmp"
        staged.write_text(json.dumps(summary,indent=2))
        staged.replace(output/"session_status.json")
    return result

def worker_pages(universe, chosen, workers):
    positions = {tuple(key): i for i, key in enumerate(universe)}
    if len(positions) != len(universe) or len(set(map(tuple,chosen))) != len(chosen):
        raise ValueError("Duplicate page IDs")
    return [[tuple(page) for page in chosen if positions[tuple(page)] % workers == i] for i in range(workers)]

def choose_tesseract_profile(output, resource, fingerprint, samples, weights, root, command_code, base_env):
    import json, random, tempfile, time
    from pathlib import Path
    cache = Path(output) / "tesseract_cpu_tuning.json"
    if cache.is_file():
        saved = json.loads(cache.read_text(encoding="utf-8"))
        if saved.get("fingerprint") == fingerprint and saved.get("profile") in cpu_candidates(resource["budget"]):
            print("[CPU] Reusing frozen calibration:", saved["profile"], flush=True)
            return saved["profile"]
        if list(Path(output).glob("tesseract-*.sqlite")):
            raise ValueError("CPU/runtime fingerprint changed. Keep old results; choose a new OUTPUT for a new pilot.")
    records = []
    candidates = cpu_candidates(resource["budget"])
    random.Random(42).shuffle(candidates)
    for profile in candidates:
        record = dict(profile=profile, error=None)
        print("[CPU CALIBRATION]", profile, flush=True)
        try:
            with tempfile.TemporaryDirectory() as td:
                start = str(Path(td)/"start")
                tasks = list(samples) * 4
                jobs = [dict(weights=weights, root=str(root), pages=tasks[i::profile["workers"]],
                             ready=str(Path(td)/f"ready{i}"), start=start,
                             result=str(Path(td)/f"result{i}")) for i in range(profile["workers"])]
                seconds = launch_cpu_jobs(jobs, command_code, profile, base_env, 600, barrier=start)
                results = [json.loads(Path(job["result"]).read_text()) for job in jobs]
                count = sum(result["pages"] for result in results)
                if count != len(tasks):
                    raise ValueError("Incomplete CPU calibration")
                record.update(pages=count, wall_seconds=seconds, pages_per_second=count/seconds)
        except Exception as exc:
            record["error"] = repr(exc)
        records.append(record)
        print("[CPU CALIBRATION RESULT]", record, flush=True)
    profile = select_cpu_profile(records)
    saved = dict(fingerprint=fingerprint, resource=resource, profile=profile, samples=samples,
                 repetitions=4, records=records, selection="maximum successful wall pages/s; no GT or OCR scores")
    staged = cache.with_suffix(".tmp")
    staged.write_text(json.dumps(saved, indent=2), encoding="utf-8")
    staged.replace(cache)
    print("[CPU] Selected frozen profile:", profile, flush=True)
    return profile



import inspect, platform, random, time
from importlib.metadata import distributions
versions = {d.metadata["Name"].lower(): d.version for d in distributions() if d.metadata["Name"]}
worker_code = ("import subprocess, sys, json\n" + inspect.getsource(load_reader) + "\n" +
               inspect.getsource(evaluate) + "\nevaluate(json.load(sys.stdin))\n")
calibration_code = ("import subprocess, sys, json\n" + inspect.getsource(load_reader) + "\n" +
                    inspect.getsource(calibrate_worker) + "\ncalibrate_worker(json.load(sys.stdin))\n")
resource = detect_cpu_budget()
cpu_name = platform.machine()
if Path("/proc/cpuinfo").exists():
    cpu_name = next((line.split(":",1)[1].strip() for line in Path("/proc/cpuinfo").read_text().splitlines()
                     if line.startswith("model name")), cpu_name)
native_version = subprocess.check_output([exe,"--version"], text=True).splitlines()[0]
samples = []
for region, count in [("text",2),("nontext",1)]:
    samples.extend(random.Random(42).sample([key for key in pages if key[0]==region],5)[:count])
fingerprint_data = dict(resource=resource, cpu=cpu_name, platform=platform.platform(), python=sys.version,
                        versions=versions, tesseract=native_version, traineddata_sha256=expected_sha,
                        reader_sha256=hashlib.sha256(worker_code.encode()).hexdigest(),
                        calibration_sha256=hashlib.sha256((calibration_code+inspect.getsource(launch_cpu_jobs)+
                                    inspect.getsource(choose_tesseract_profile)+inspect.getsource(cpu_process_env)).encode()).hexdigest(),
                        samples=samples, repetitions=4, candidates=cpu_candidates(resource["budget"]))
fingerprint = hashlib.sha256(json.dumps(fingerprint_data,sort_keys=True).encode()).hexdigest()
print("[CPU] Detected resources:", resource, flush=True)
import fcntl
lock_stream = (OUTPUT / ".tesseract.lock").open("a+")
try:
    fcntl.flock(lock_stream.fileno(), fcntl.LOCK_EX | fcntl.LOCK_NB)
except BlockingIOError:
    lock_stream.close()
    raise RuntimeError("Another Tesseract run is active in this OUTPUT. Stop it before Run All.")
try:
    if RESUME_FROM:
        source = Path(RESUME_FROM)
        source_dir = source if source.is_dir() else source.parent
        cached = source_dir / "tesseract_cpu_tuning.json"
        target = OUTPUT / cached.name
        if not cached.is_file():
            raise FileNotFoundError("Tuned resume requires tesseract_cpu_tuning.json beside the worker SQLite files.")
        if not target.exists():
            incoming = json.loads(cached.read_text(encoding="utf-8"))
            if incoming.get("fingerprint") != fingerprint:
                raise ValueError("Input CPU calibration/runtime differs. Use a fresh OUTPUT and pilot.")
            staged = target.with_suffix(".tmp")
            staged.write_bytes(cached.read_bytes()); staged.replace(target)
    profile = choose_tesseract_profile(OUTPUT, resource, fingerprint, samples, weights, ROOT, calibration_code, runtime_env)
    workers, threads = profile["workers"], profile["threads"]
    config = dict(model=MODEL, dataset=DATA_REV, revisions=revisions, weights_mib=weight_mib,
                  prompt="", workers=workers, versions=versions, cpu=cpu_name, threads=threads,
                  reader_sha256=hashlib.sha256(worker_code.encode()).hexdigest(), timing_mode="single_page",
                  oem=1, psm=3, language="eng", traineddata_sha256=expected_sha, platform=platform.platform(),
                  tesseract=native_version, cpu_parallelism="independent page processes; shared host resources",
                  thread_limits="OMP/MKL/OPENBLAS/NUMEXPR=threads_per_worker; nested OMP disabled")
    chunks = worker_pages(pages, selected, workers)
    all_chunks = worker_pages(pages, pages, workers)
    jobs = []
    for index, chunk in enumerate(chunks):
        job = dict(model=MODEL,pages=chunk,gpu=False,config=config,output=str(OUTPUT),root=str(ROOT),
                   weights=weights,prompt="",worker=index)
        digest = hashlib.sha256(json.dumps(dict(config,device="CPU"),sort_keys=True).encode()).hexdigest()[:16]
        if RESUME_FROM:
            if source.is_file() and workers != 1:
                raise ValueError("Parallel resume requires a directory containing all worker SQLite files and tuning JSON.")
            path = source/f"tesseract-{digest}-{index}.sqlite" if source.is_dir() else source
            if not path.is_file():
                raise FileNotFoundError(f"Missing compatible worker checkpoint: {path}")
            restore_checkpoint(path, OUTPUT, dict(config,device="CPU"), all_chunks[index], index)
        working = OUTPUT/f"tesseract-{digest}-{index}.sqlite"
        if working.is_file():
            restore_checkpoint(working, OUTPUT, dict(config,device="CPU"), all_chunks[index], index)
        jobs.append(job)
    print(f"[RUN] {workers} workers x {threads} threads; {len(selected)} pages; batch=1",flush=True)
    status = supervise_cpu_jobs(jobs, worker_code, profile, runtime_env, OUTPUT,
                               time.monotonic()+11*3600, max_restarts=2, stall_seconds=600)
    print("[RUN STATUS]",status,flush=True)
finally:
    try:
        export_checkpoints(OUTPUT)
    finally:
        fcntl.flock(lock_stream.fileno(), fcntl.LOCK_UN)
        lock_stream.close()


### Score saved predictions

Macro means by page. Pilot/partial tables are provisional; full completion requires exactly 6,000 + 3,000 successful IDs. Run the shared `evaluate_oprb.py` locally on downloaded SQLite files for the final report.


In [ ]:
%%benchmark
from contextlib import closing
def occlusion_metadata(stem):
    import re
    parts = stem.split("__", 2)
    density = parts[1] if len(parts) > 1 and re.fullmatch(r"[0-9]+(?:p[0-9]+)?pct", parts[1]) else "unknown"
    return parts[0], density

def score_results():
    rows = []
    refs = {}
    for path in sorted(OUTPUT.glob(f"{MODEL}-*.sqlite")):
        with closing(sqlite3.connect(path)) as db:
            cfg, = db.execute("SELECT json FROM config").fetchone()
            config = json.loads(cfg)
            if config["dataset"] != DATA_REV:
                continue
            for region, stem, pred, seconds, trunc, error in db.execute("SELECT * FROM pages"):
                key = (region, stem)
                if key not in refs:
                    refs[key] = reference(*key)
                scores = (None,)*5 if error else metrics(refs[key], pred)
                parts = occlusion_metadata(stem)
                rows.append((config["model"], hashlib.sha256(cfg.encode()).hexdigest()[:12], config["device"],
                             config["workers"], config["weights_mib"], region, stem, parts[0], parts[1],
                             pred, refs[key], seconds, trunc, error, *scores))
    cols = ["model","config","device","workers","weights_mib","region","stem","type","density",
            "prediction","gt","seconds","truncated","error","cer","wer","precision","recall","f1"]
    return pd.DataFrame(rows, columns=cols)

results = score_results()
assert len(results) and not results.duplicated(["model","config","region","stem"]).any()
keys = ["model","config"]
values = ["cer","wer","precision","recall","f1","seconds"]
coverage = results.groupby(keys + ["region"]).agg(attempted=("stem","size"), ok=("seconds","count"),
                                                   truncated=("truncated","sum"))
coverage["expected"] = coverage.index.get_level_values("region").map(COUNTS)
display(coverage)
ok = results[results.error.isna()]
display(ok.groupby(keys+["device","weights_mib"])[values].mean())
for groups in (["region"], ["region","type"], ["region","density"]):
    display(ok.groupby(keys+groups)[values].mean().join(ok.groupby(keys+groups).size().rename("pages")))
latency = ok.groupby(keys+["workers","region"]).seconds.mean().unstack().reindex(columns=list(COUNTS))
display((latency.mul(pd.Series(COUNTS)).sum(axis=1, skipna=False) /
         (3600 * latency.index.get_level_values("workers"))).rename("projected_wall_hours"))
display(pd.DataFrame([(r, *occlusion_metadata(s)) for r,s in pages],
                     columns=["region","type","density"]).value_counts().rename("dataset_pages"))
# Locate qualitative candidates. Inspect the same image crop, GT and predictions before using them.
for region, group in ok.groupby("region"):
    print(region)
    display(pd.concat([group.nsmallest(1,"wer"),group.nlargest(1,"wer")])[
        ["model","stem","wer","gt","prediction"]])
# Example: display(Image.open(ROOT / region / "img" / (stem + ".jpg")))